# `03_attrition_check.ipynb`

This notebook is the fourth step in the pipeline. It assesses survey attrition in the consented cohort by:
- Anchoring on the Qualtrics “BEFORE” survey roster
- Mapping consent/DSC status from the “AFTER” survey
- Deriving baseline characteristics
- Running group comparisons (Welch t-tests, χ² tests with Holm correction)
- Exporting LaTeX tables and parquet extracts to the Research Drive

### What it does
- Loads Qualtrics “before” and “after” CSV exports (skipping the first two metadata rows that Qualtrics inserts).
- Applies a start-date cutoff (CUTOFF_DATE = 2024-10-14) to restrict to the relevant field period.
- Anchors the consented cohort on unique uids from the BEFORE survey; maps “dsc” consent from AFTER when available.
- Derives a set of baseline variables from BEFORE (if present):
  - Age (handling Year-of-Birth variants), Gender harmonization
  - Etniciteit_binair, Werk_binair
  - Education recodes (Edu_breed, Edu_binary)
  - Political orientation (PolOrientation_cat)
  - Issue importance/knowledge (ImportIssue_1_numeric, KnowIssue_1_numeric)
  - Attitude scales (AttitudeExtr1_combined, AttitudeExtr2_combined)
- Compares “Dropout (no dsc)” vs “With dsc” groups on baseline variables:
  - Continuous: Welch t-tests with means/SDs and Holm-adjusted p-values
  - Categorical: omnibus χ² tests with Holm-adjusted p-values and level-wise distributions
- Classifies attrition more broadly (requires annotated_any provided earlier in the pipeline):
  - Dropout_no_dsc, Dropout_no_annotations, Completer
- Compares “Dropouts” vs “Completers” on the same baseline variables, with parallel LaTeX + parquet outputs.
- Uploads all outputs to the Research Drive via WebDAV utilities.

### Inputs
- Qualtrics exports:
  - `data/Qualtrics_data_exports/before.csv`
  - `data/Qualtrics_data_exports/after.csv`
- Configuration and utilities:
  - `config.PROJECT_ROOT` (used to resolve project-relative paths)
  - `rd_utils` (WebDAV-enabled IO helpers: read_csv, write_parquet, webdav_mkdirs, webdav_upload_bytes)
- Columns expected in Qualtrics:
  - `uid` (key), `StartDate` (for cutoff)
  - `dsc` (consent indicator) expected in AFTER
  - Optional baseline columns listed above (derived when present)
- For the broader attrition comparison:
  - `annotated_any` must already exist on the consented roster (from earlier data-preparation steps).

### Key parameters and checks
- Start-date cutoff: `CUTOFF_DATE = 2024-10-14`
- Qualtrics read: `skiprows=[1, 2]` to strip metadata lines
- Anchoring on BEFORE for the consented roster and baselines
- Welch t-tests for continuous variables; χ² for categorical
- Multiple-testing correction: Holm method
- Robust LaTeX escaping to ensure compilable tables

### Outputs
LaTeX tables (Research Drive, under `output/tables`):
- Dropout vs With dsc
  - `output/tables/dropout_no_dsc_continuous.tex`
  - `output/tables/dropout_no_dsc_categorical.tex`
  - `output/tables/dropout_no_dsc_categorical_detail.tex`
- Attrition (Dropouts vs Completers)
  - `output/tables/attrition_continuous_from_before.tex`
  - `output/tables/attrition_categorical_from_before.tex`
  - `output/tables/attrition_categorical_detail_from_before.tex`

Parquet extracts (Research Drive, under `output/derived`):
- Dropout vs With dsc
  - `output/derived/consented_with_dsc_baseline.parquet`
  - `output/derived/dropouts_no_dsc_from_before.parquet`
  - `output/derived/with_dsc_from_before.parquet`
- Attrition (Dropouts vs Completers)
  - `output/derived/consented_attrition_with_baseline.parquet`
  - `output/derived/completers_from_before.parquet`
  - `output/derived/dropouts_no_dsc_from_before.parquet`
  - `output/derived/dropouts_no_annotations_from_before.parquet`

### Console summaries
The notebook prints:
- Counts for Completers, Dropout_no_dsc, Dropout_no_annotations within the consented cohort
- Continuous comparison table preview (means/SDs, t, p, Holm-adjusted p)
- Categorical omnibus χ² summary
- Level-wise percentage distributions for key categorical variables



In [1]:
from datetime import datetime
from io import BytesIO

import numpy as np
import pandas as pd

import config
import rd_utils as rd
from rd_utils import webdav_mkdirs, webdav_upload_bytes

# ---------------------------
# Paths (relative to project root)
# ---------------------------
PROJ       = config.PROJECT_ROOT
QUALTRICS  = f"{PROJ}/data/Qualtrics_data_exports"
AT_EXPORTS = f"{PROJ}/data/AnnoTinder_data_exports"

TABLES_DIR  = f"{PROJ}/output/tables"
DERIVED_DIR = f"{PROJ}/output/derived"

webdav_mkdirs(TABLES_DIR)
webdav_mkdirs(DERIVED_DIR)

CUTOFF_DATE = datetime(2024, 10, 14)

ANNOTINDER_FILES = [
    "annotations_95_AnnoBias job final, set 1.csv.csv",
    "annotations_96_AnnoBias job final, set 2.csv.csv",
    "annotations_97_AnnoBias job final, set 3.csv.csv",
    "annotations_98_AnnoBias job final, set 4.csv.csv",
    "annotations_99_AnnoBias job final, set 5.csv.csv",
    "annotations_100_AnnoBias job final, set 6.csv.csv",
]

# ---------------------------
# IO helpers (via rd_utils)
# ---------------------------
def read_qx(folder: str, fname: str) -> pd.DataFrame:
    """Read Qualtrics CSV export (known header pattern) from Research Drive."""
    return rd.read_csv(f"{folder}/{fname}", low_memory=False, skiprows=[1, 2])

def apply_cutoff(df: pd.DataFrame, cutoff: datetime) -> pd.DataFrame:
    """Keep rows with StartDate >= cutoff (if StartDate exists)."""
    if "StartDate" not in df.columns:
        return df
    dt = pd.to_datetime(df["StartDate"], errors="coerce")
    try:
        if getattr(dt.dt, "tz", None) is not None:
            dt = dt.dt.tz_localize(None)
    except Exception:
        pass
    keep = dt >= cutoff
    return df.loc[keep.fillna(False)].copy()

def upload_text(rel_path: str, text: str):
    webdav_upload_bytes(rel_path, text.encode("utf-8"), content_type="text/plain")
    print(f"✅ Uploaded {rel_path}")

def upload_parquet(rel_path: str, df: pd.DataFrame):
    rd.write_parquet(df, rel_path)
    print(f"✅ Uploaded {rel_path}")

# ---------------------------
# Minimal LaTeX builder (manual)
# ---------------------------
def latex_escape(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    return (s.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")
             .replace("#", r"\#").replace("$", r"\$").replace("{", r"\{")
             .replace("}", r"\}").replace("~", r"\textasciitilde{}")
             .replace("^", r"\textasciicircum{}"))

def df_to_latex_tabular(df: pd.DataFrame, caption: str, label: str) -> str:
    df_str = df.copy()
    for c in df_str.columns:
        df_str[c] = df_str[c].apply(lambda x: "" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x))

    headers = " & ".join(latex_escape(c) for c in df_str.columns)
    colspec = "l" + "c" * (df_str.shape[1] - 1)
    rows = [" & ".join(row) + r" \\" for row in df_str.values.tolist()]
    body = "\n".join(rows)

    return (
        r"\begin{table}[ht]" "\n"
        r"\centering" "\n"
        f"\\caption{{{latex_escape(caption)}}}\n"
        f"\\label{{{latex_escape(label)}}}\n"
        "\\begin{tabular}{" + colspec + "}\n"
        r"\toprule" "\n"
        + headers + r" \\" "\n"
        r"\midrule" "\n"
        + body + "\n"
        r"\bottomrule" "\n"
        r"\end{tabular}" "\n"
        r"\end{table}"
    )

# ---------------------------
# Load Qualtrics BEFORE / AFTER
# ---------------------------
before = read_qx(QUALTRICS, "before.csv")
after  = read_qx(QUALTRICS, "after.csv")

before.columns = [c.strip() for c in before.columns]
after.columns  = [c.strip() for c in after.columns]

before = apply_cutoff(before, CUTOFF_DATE)
after  = apply_cutoff(after,  CUTOFF_DATE)

print(f"[Load] Qualtrics after cutoff: before={len(before)} | after={len(after)}")

if "uid" not in before.columns:
    raise KeyError("before.csv missing required column 'uid' (after skiprows=[1,2]).")
if "uid" not in after.columns:
    raise KeyError("after.csv missing required column 'uid' (after skiprows=[1,2]).")

has_dsc = "dsc" in after.columns
print(f"[Load] Found dsc in after.csv: {has_dsc}")

# ---------------------------
# Build consented cohort (anchored on BEFORE)
# ---------------------------
consented = (
    before[["uid"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"uid": "id_key"})
)
print(f"[Cohort] Consented unique uids (before): {len(consented)}")

# Map dsc from AFTER
if has_dsc:
    dsc_map = (
        after[["uid", "dsc"]]
        .dropna(subset=["uid"])
        .drop_duplicates(subset=["uid"])
        .set_index("uid")["dsc"]
    )
    consented["dsc"] = consented["id_key"].map(dsc_map)
else:
    consented["dsc"] = np.nan

consented["has_dsc"] = consented["dsc"].notna().astype(int)

# ---------------------------
# Compute annotated_any from AnnoTinder exports (explicit + robust)
# ---------------------------
consented = consented.copy()
consented["id_key_str"] = consented["id_key"].astype(str).str.strip()

annot_uids = set()
for fname in ANNOTINDER_FILES:
    tmp = rd.read_csv(f"{AT_EXPORTS}/{fname}")
    if "coder" not in tmp.columns:
        raise KeyError(f"[AnnoTinder] {fname} missing required 'coder' column.")
    annot_uids.update(tmp["coder"].astype(str).str.strip().dropna().unique())

consented["annotated_any"] = consented["id_key_str"].isin(annot_uids).astype(int)

overlap_n = int(consented["annotated_any"].sum())
print(f"[AnnoTinder] unique coder uids: {len(annot_uids)}")
print(f"[AnnoTinder] consented with annotations: {overlap_n} ({overlap_n/len(consented):.1%})")

if overlap_n == 0:
    print("\n[DIAG] No overlap found between Qualtrics uid and AnnoTinder coder.")
    print("  Qualtrics uid examples:", consented["id_key_str"].dropna().unique()[:5])
    print("  AnnoTinder uid examples:", list(sorted(annot_uids))[:5])
    raise ValueError("annotated_any is 0 for everyone. You are likely joining on the wrong identifier (uid mismatch).")

# ---------------------------
# Derive baselines (from BEFORE)
# ---------------------------
demo = before.copy()
cur_year = datetime.now().year

# Age (Age or Age_qualtrics where Age_qualtrics holds YOB)
if "Age" in demo.columns:
    demo["Age"] = pd.to_numeric(demo["Age"], errors="coerce")
    mask_yob = demo["Age"] > 1900
    demo.loc[mask_yob, "Age"] = cur_year - demo.loc[mask_yob, "Age"]
elif "Age_qualtrics" in demo.columns:
    yob = pd.to_numeric(demo["Age_qualtrics"], errors="coerce")
    demo["Age"] = cur_year - yob

# Gender harmonization
if "Gender" not in demo.columns and "gender" in demo.columns:
    demo["Gender"] = demo["gender"].map({"male": "Man", "female": "Vrouw"}).fillna("Anders")

# Ethnicity binary
if "Etniciteit" in demo.columns and "Etniciteit_binair" not in demo.columns:
    demo["Etniciteit_binair"] = demo["Etniciteit"].apply(lambda x: 1 if str(x).strip() == "Nederlands" else 0)

# Work binary
if "Werk" in demo.columns and "Werk_binair" not in demo.columns:
    demo["Werk_binair"] = demo["Werk"].apply(
        lambda x: 1 if x in ["Werkend (betaalde werknemer)", "Werkend (zelfstandig ondernemer)"] else 0
    )

# Education recodes
if "Edu" in demo.columns:
    def recode_edu(v):
        if v in ["Geen onderwijs gevolgd of het niet afgemaakt", "Lagere school (basisonderwijs)"]:
            return "Basisonderwijs"
        if v in ["Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)", "Anders, namelijk:"]:
            return "Voortgezet onderwijs"
        if v in [
            "Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)",
            "Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)",
            "Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)",
            "Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)",
        ]:
            return "Praktijkopleiding"
        if v in ["Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)",
                 "Wetenschappelijk onderwijs (universiteit)"]:
            return "Hoger onderwijs"
        return np.nan

    demo["Edu_breed"] = pd.Categorical(
        demo["Edu"].apply(recode_edu),
        categories=["Basisonderwijs", "Voortgezet onderwijs", "Praktijkopleiding", "Hoger onderwijs"],
        ordered=True
    )

# Political orientation
if "PolOrient_1" in demo.columns:
    def recode_pol(v):
        try:
            v = float(v)
        except Exception:
            return np.nan
        if v <= 3:
            return "Links"
        if v <= 6:
            return "Midden"
        if v <= 10:
            return "Rechts"
        return np.nan
    demo["PolOrientation_cat"] = demo["PolOrient_1"].apply(recode_pol)

# Issues (numeric)
imp_map = {"Zeer belangrijk": 4, "Belangrijk": 3, "Niet belangrijk, maar ook niet onbelangrijk": 2, "Onbelangrijk": 1, "Zeer onbelangrijk": 0}
knw_map = {"Heel veel": 4, "Veel": 3, "Niet veel, maar ook niet weinig": 2, "Weinig": 1, "Heel weinig": 0}
if "ImportIssue_1" in demo.columns:
    demo["ImportIssue_1_numeric"] = demo["ImportIssue_1"].map(imp_map)
if "KnowIssue_1" in demo.columns:
    demo["KnowIssue_1_numeric"] = demo["KnowIssue_1"].map(knw_map)

# Attitudes (combined)
def rec_att(v):
    return {"Zeer mee eens": 4, "Mee eens": 3, "Niet eens, maar ook niet oneens": 2, "Mee oneens": 1, "Zeer mee oneens": 0}.get(v, np.nan)
def rec_att_rev(v):
    return {"Zeer mee eens": 0, "Mee eens": 1, "Niet eens, maar ook niet oneens": 2, "Mee oneens": 3, "Zeer mee oneens": 4}.get(v, np.nan)

if set(["AttitudeExtr2_1", "AttitudeExtr2_2", "AttitudeExtr2_3", "AttitudeExtr2_4"]).issubset(demo.columns):
    demo["AttitudeExtr2_1_numeric"] = demo["AttitudeExtr2_1"].apply(rec_att)
    demo["AttitudeExtr2_2_numeric"] = demo["AttitudeExtr2_2"].apply(rec_att)
    demo["AttitudeExtr2_3_numeric"] = demo["AttitudeExtr2_3"].apply(rec_att_rev)
    demo["AttitudeExtr2_4_numeric"] = demo["AttitudeExtr2_4"].apply(rec_att)
    demo["AttitudeExtr2_combined"] = demo[
        ["AttitudeExtr2_1_numeric","AttitudeExtr2_2_numeric","AttitudeExtr2_3_numeric","AttitudeExtr2_4_numeric"]
    ].mean(axis=1)

if set(["AttitudeExtr1_1", "AttitudeExtr1_2", "AttitudeExtr1_3", "AttitudeExtr1_4"]).issubset(demo.columns):
    demo["AttitudeExtr1_1_numeric"] = demo["AttitudeExtr1_1"].apply(rec_att)
    demo["AttitudeExtr1_2_numeric"] = demo["AttitudeExtr1_2"].apply(rec_att)
    demo["AttitudeExtr1_3_numeric"] = demo["AttitudeExtr1_3"].apply(rec_att)
    demo["AttitudeExtr1_4_numeric"] = demo["AttitudeExtr1_4"].apply(rec_att_rev)
    demo["AttitudeExtr1_combined"] = demo[
        ["AttitudeExtr1_1_numeric","AttitudeExtr1_2_numeric","AttitudeExtr1_3_numeric","AttitudeExtr1_4_numeric"]
    ].mean(axis=1)

baseline_cols = [c for c in [
    "Age","Gender","Etniciteit_binair","Werk_binair","Edu_breed",
    "PolOrientation_cat","ImportIssue_1_numeric","KnowIssue_1_numeric",
    "AttitudeExtr1_combined","AttitudeExtr2_combined"
] if c in demo.columns]

incoming = demo[["uid"] + baseline_cols].drop_duplicates("uid").rename(columns={"uid":"id_key"})
consented = consented.merge(incoming, on="id_key", how="left")
print(f"[Baseline] variables available: {baseline_cols}")

# ---------------------------
# Attrition grouping
# ---------------------------
def classify_attrition(r):
    if r["has_dsc"] == 0:
        return "Dropout_no_dsc"
    if r["annotated_any"] == 0:
        return "Dropout_no_annotations"
    return "Completer"

consented["attrition_group"] = consented.apply(classify_attrition, axis=1)
consented["is_dropout"] = (consented["attrition_group"] != "Completer").astype(int)

print("\n[Groups] attrition_group counts:")
print(consented["attrition_group"].value_counts(dropna=False).to_string())

# ---------------------------
# Stats helpers (Welch t, chi-square, Holm)
# ---------------------------
try:
    import scipy.stats as st
except Exception:
    st = None
try:
    from statsmodels.stats.multitest import multipletests
except Exception:
    multipletests = None

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return r"<.001"
    return f"{p:.3f}".lstrip("0")

def welch_t(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    y = pd.to_numeric(pd.Series(y), errors="coerce").dropna()
    if len(x) < 2 or len(y) < 2 or st is None:
        return (np.nan, np.nan, np.nan)
    mx, my = x.mean(), y.mean()
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    nx, ny = len(x), len(y)
    den = np.sqrt(vx/nx + vy/ny)
    if den == 0:
        return (np.nan, np.nan, np.nan)
    t = (mx - my) / den
    dfw = (vx/nx + vy/ny)**2 / ((vx**2)/(nx**2*(nx-1)) + (vy**2)/(ny**2*(ny-1)))
    p = 2 * st.t.sf(np.abs(t), dfw)
    return (t, dfw, p)

def chi2_test(cat, grp):
    tmp = pd.DataFrame({"c": cat, "g": grp}).dropna()
    if tmp.empty or tmp["c"].nunique() < 2 or tmp["g"].nunique() < 2 or st is None:
        return (np.nan, np.nan, np.nan, pd.DataFrame())
    tab = pd.crosstab(tmp["g"], tmp["c"])
    chi2v, p, dof, _ = st.chi2_contingency(tab)
    return (chi2v, dof, p, tab)

def percent(n, d):
    return 0.0 if d == 0 else 100.0 * n / d

# Variables to compare (baseline only)
continuous_vars = [v for v in [
    "Age","ImportIssue_1_numeric","KnowIssue_1_numeric",
    "AttitudeExtr1_combined","AttitudeExtr2_combined"
] if v in consented.columns]

categorical_vars = [v for v in [
    "Gender","Etniciteit_binair","Edu_breed","PolOrientation_cat","Werk_binair"
] if v in consented.columns]

# ---------------------------
# Continuous comparisons: dropouts vs completers
# ---------------------------
mask_c = consented["is_dropout"] == 0
mask_d = consented["is_dropout"] == 1

cont_rows = []
for v in continuous_vars:
    x_c = pd.to_numeric(consented.loc[mask_c, v], errors="coerce")
    x_d = pd.to_numeric(consented.loc[mask_d, v], errors="coerce")
    mC, sC = x_c.mean(), x_c.std(ddof=1)
    mD, sD = x_d.mean(), x_d.std(ddof=1)
    t, dfw, p = welch_t(x_d, x_c)  # Dropouts vs Completers
    cont_rows.append({
        "Variable": v,
        "Dropouts M (SD)": f"{mD:.2f} ({sD:.2f})",
        "Completers M (SD)": f"{mC:.2f} ({sC:.2f})",
        "t (df)": f"{t:.2f} ({dfw:.1f})" if np.isfinite(t) and np.isfinite(dfw) else "",
        "p_raw": p
    })

cont_df = pd.DataFrame(cont_rows)
if multipletests is not None and cont_df["p_raw"].notna().any():
    m = cont_df["p_raw"].notna()
    cont_df.loc[m, "p_adj_raw"] = multipletests(cont_df.loc[m, "p_raw"], method="holm")[1]
cont_df["p"] = cont_df["p_raw"].apply(fmt_p)
if "p_adj_raw" in cont_df.columns:
    cont_df["p_adj (Holm)"] = cont_df["p_adj_raw"].apply(fmt_p)
cont_df = cont_df.drop(columns=[c for c in ["p_raw","p_adj_raw"] if c in cont_df.columns])

# ---------------------------
# Categorical: omnibus χ² + detail block
# ---------------------------
cat_sum_rows, cat_blocks = [], []
for v in categorical_vars:
    chi2v, dof, p, tab = chi2_test(consented[v], consented["is_dropout"])
    cat_sum_rows.append({
        "Variable": v,
        r"$\chi^2$(df)": f"{chi2v:.2f} ({int(dof)})" if np.isfinite(chi2v) else "",
        "p_raw": p
    })

    d_counts = consented.loc[mask_d, v].value_counts(dropna=False)
    c_counts = consented.loc[mask_c, v].value_counts(dropna=False)
    levels = sorted(set(d_counts.index).union(set(c_counts.index)), key=lambda x: str(x))
    denom_d = d_counts.drop(labels=[np.nan], errors="ignore").sum()
    denom_c = c_counts.drop(labels=[np.nan], errors="ignore").sum()

    lines = [r"\midrule", rf"\textbf{{{latex_escape(v)}}} & \textbf{{Dropouts}} & \textbf{{Completers}} \\"]
    for lvl in levels:
        if pd.isna(lvl):
            continue
        nD = int(d_counts.get(lvl, 0)); nC = int(c_counts.get(lvl, 0))
        lines.append(r"\quad " + f"{latex_escape(lvl)} & {nD} ({percent(nD, denom_d):.1f}\\%) & {nC} ({percent(nC, denom_c):.1f}\\%) \\\\")
    lines.append(
        r"\quad " + (fr"\emph{{Test}} & $\chi^2({int(dof)})={chi2v:.2f},\, p={fmt_p(p)}$ & \\"
                     if np.isfinite(chi2v) else r"\emph{Test} & & \\")
    )
    cat_blocks.append("\n".join(lines))

cat_summary_df = pd.DataFrame(cat_sum_rows)
if multipletests is not None and cat_summary_df["p_raw"].notna().any():
    m = cat_summary_df["p_raw"].notna()
    cat_summary_df.loc[m, "p_adj_raw"] = multipletests(cat_summary_df.loc[m, "p_raw"], method="holm")[1]
cat_summary_df["p"] = cat_summary_df["p_raw"].apply(fmt_p)
if "p_adj_raw" in cat_summary_df.columns:
    cat_summary_df["p_adj (Holm)"] = cat_summary_df["p_adj_raw"].apply(fmt_p)
cat_summary_df = cat_summary_df.drop(columns=[c for c in ["p_raw","p_adj_raw"] if c in cat_summary_df.columns])

# ---------------------------
# Build LaTeX strings
# ---------------------------
cont_cols = ["Variable","Dropouts M (SD)","Completers M (SD)","t (df)","p"]
if "p_adj (Holm)" in cont_df.columns:
    cont_cols.append("p_adj (Holm)")
cont_tex = df_to_latex_tabular(
    cont_df[cont_cols],
    caption="Attrition (consented cohort): continuous baseline variables, dropouts vs. completers",
    label="tab:attrition_continuous_from_before"
)

cat_cols = ["Variable", r"$\chi^2$(df)", "p"]
if "p_adj (Holm)" in cat_summary_df.columns:
    cat_cols.append("p_adj (Holm)")
cat_sum_tex = df_to_latex_tabular(
    cat_summary_df[cat_cols],
    caption="Attrition (consented cohort): categorical baseline variables (omnibus)",
    label="tab:attrition_categorical_from_before"
)

cat_detail_tex = (
    r"\begin{table}[ht]" "\n"
    r"\centering" "\n"
    r"\caption{Attrition (consented cohort): categorical baseline variables, level-wise distributions}" "\n"
    r"\label{tab:attrition_categorical_detail_from_before}" "\n"
    r"\begin{tabular}{lcc}" "\n"
    r"\toprule" "\n"
    r"\textbf{Level} & \textbf{Dropouts} & \textbf{Completers} \\" "\n"
    + "\n".join(cat_blocks) + "\n"
    r"\bottomrule" "\n"
    r"\end{tabular}" "\n"
    r"\end{table}"
)

# ---------------------------
# Upload outputs
# ---------------------------
upload_text(f"{TABLES_DIR}/attrition_continuous_from_before.tex", cont_tex)
upload_text(f"{TABLES_DIR}/attrition_categorical_from_before.tex", cat_sum_tex)
upload_text(f"{TABLES_DIR}/attrition_categorical_detail_from_before.tex", cat_detail_tex)

upload_parquet(f"{DERIVED_DIR}/consented_attrition_with_baseline.parquet", consented)
upload_parquet(f"{DERIVED_DIR}/completers_from_before.parquet", consented.loc[consented["attrition_group"] == "Completer"])
upload_parquet(f"{DERIVED_DIR}/dropouts_no_dsc_from_before.parquet", consented.loc[consented["attrition_group"] == "Dropout_no_dsc"])
upload_parquet(f"{DERIVED_DIR}/dropouts_no_annotations_from_before.parquet", consented.loc[consented["attrition_group"] == "Dropout_no_annotations"])

# ---------------------------
# Inline console summary (correct)
# ---------------------------
grp_counts = consented["attrition_group"].value_counts(dropna=False).to_dict()
n_cons = len(consented)
n_comp = int(grp_counts.get("Completer", 0))
n_ndsc = int(grp_counts.get("Dropout_no_dsc", 0))
n_nann = int(grp_counts.get("Dropout_no_annotations", 0))

print("\n=== Attrition (within consented cohort; anchored on BEFORE) ===")
print(f"Consented (N): {n_cons}")
print(f"  • Completer (has_dsc & annotated_any):        {n_comp}")
print(f"  • Dropout_no_dsc (no valid dsc):              {n_ndsc}")
print(f"  • Dropout_no_annotations (has_dsc, no ann.):  {n_nann}")

print("\n=== Continuous comparisons (Dropouts vs Completers) ===")
print(cont_df[cont_cols].fillna("").to_string(index=False))

print("\n=== Categorical omnibus (Dropouts vs Completers) ===")
print(cat_summary_df[cat_cols].fillna("").to_string(index=False))

print("\n✅ All attrition outputs uploaded to Research Drive")

[Load] Qualtrics after cutoff: before=2163 | after=1584
[Load] Found dsc in after.csv: True
[Cohort] Consented unique uids (before): 2162
[AnnoTinder] unique coder uids: 1461
[AnnoTinder] consented with annotations: 1461 (67.6%)
[Baseline] variables available: ['Age', 'Gender', 'Etniciteit_binair', 'Werk_binair', 'Edu_breed', 'PolOrientation_cat', 'ImportIssue_1_numeric', 'KnowIssue_1_numeric', 'AttitudeExtr1_combined', 'AttitudeExtr2_combined']

[Groups] attrition_group counts:
attrition_group
Completer                 1386
Dropout_no_dsc             687
Dropout_no_annotations      89
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/attrition_continuous_from_before.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/attrition_categorical_from_before.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/attrition_categorical_detail_from_before.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/derived/consented_attrition_with_b